In [ ]:
workspace_id = "c72720ab-d6b2-414f-9ce5-bff9817f7946"
warehouse_id = "03a2aa43-0293-4584-a250-1fac3af41175"
staged_file = "/lakehouse/default/Files/property-leading-indicators/prepared.json"


In [ ]:
import json
from datetime import date, datetime
from decimal import Decimal
from notebookutils import data, notebook

TABLES = {
    "geographies": (["id", "country", "state", "city", "suburb", "postcode", "geography_type", "active"], "geographies"),
    "indicators": (["id", "code", "name", "category", "lead_lag", "default_frequency", "unit", "higher_is", "notes"], "indicators"),
    "property_observations": (["id", "geography_id", "indicator_id", "country", "suburb", "city", "state", "postcode", "geography_type", "indicator_code", "indicator_name", "category", "lead_lag", "unit", "higher_is", "source_name", "access_type", "period_start", "period_end", "observed_at", "value", "raw_value", "confidence", "frequency", "source_url", "notes"], "observations"),
    "fetch_runs": (["started_at", "finished_at", "status", "rows_inserted", "message"], "fetchRuns"),
    "source_register": (["source_id", "source_name", "class", "status", "access", "frequency", "geography", "indicators_json", "source_url", "notes"], "sourceRegister"),
}

FIELD_NAMES = {
    "geographies": ["id", "country", "state", "city", "suburb", "postcode", "geographyType", "active"],
    "indicators": ["id", "code", "name", "category", "leadLag", "defaultFrequency", "unit", "higherIs", "notes"],
    "property_observations": ["id", "geographyId", "indicatorId", "country", "suburb", "city", "state", "postcode", "geographyType", "indicatorCode", "indicatorName", "category", "leadLag", "unit", "higherIs", "sourceName", "accessType", "periodStart", "periodEnd", "observedAt", "value", "rawValue", "confidence", "frequency", "sourceUrl", "notes"],
    "fetch_runs": ["startedAt", "finishedAt", "status", "rowsInserted", "message"],
    "source_register": ["sourceId", "sourceName", "class", "status", "access", "frequency", "geography", "indicators", "sourceUrl", "notes"],
}

def value_for(table, field, value):
    if table == "source_register" and field == "indicators":
        return json.dumps(value or [])
    if value == "" or value is None:
        return None
    if field in {"periodStart", "periodEnd"}:
        return date.fromisoformat(str(value)[:10])
    if field in {"observedAt", "startedAt", "finishedAt"}:
        return datetime.fromisoformat(str(value).replace("Z", "+00:00")).replace(tzinfo=None)
    if field in {"value", "rawValue"}:
        return Decimal(str(value))
    return value

with open(staged_file, "r", encoding="utf-8") as handle:
    payload = json.load(handle)

expected_rows = len(payload["observations"])
expected_max_date = max(str(row["periodEnd"])[:10] for row in payload["observations"])
if expected_rows == 0:
    raise ValueError("Refusing to replace the Warehouse with zero observations")

conn = data.connect_to_artifact(warehouse_id, workspace_id, "Warehouse")
cursor = conn.cursor()
try:
    for table in reversed(list(TABLES)):
        cursor.execute(f"DELETE FROM dbo.{table}")
    for table, (columns, payload_key) in TABLES.items():
        fields = FIELD_NAMES[table]
        rows = [tuple(value_for(table, field, row.get(field)) for field in fields) for row in payload[payload_key]]
        if rows:
            placeholders = ",".join("?" for _ in columns)
            column_sql = ",".join(f"[{column}]" for column in columns)
            cursor.executemany(f"INSERT INTO dbo.{table} ({column_sql}) VALUES ({placeholders})", rows)
    cursor.execute("SELECT COUNT_BIG(*), CONVERT(varchar(10), MAX(period_end), 23) FROM dbo.property_observations")
    actual_rows, actual_max_date = cursor.fetchone()
    if int(actual_rows) != expected_rows or actual_max_date != expected_max_date:
        raise ValueError(f"Warehouse validation failed: expected {expected_rows}/{expected_max_date}, got {actual_rows}/{actual_max_date}")
    conn.commit()
except Exception:
    conn.rollback()
    raise
finally:
    cursor.close()
    conn.close()

notebook.exit(json.dumps({"status": "Completed", "rows": expected_rows, "maxDate": expected_max_date}))
